# Gradient Checking with Finite Differences

**Topic:** Matrix Calculus and Backprop Derivation &nbsp;·&nbsp; **Difficulty:** Hard

## The data (created in the next cell)

- A fixed batch `X (200, 10)`, targets `y (200,)`.
- Parameters `params = {"W1": (10, 16), "b1": (16,), "w2": (16,), "b2": ()}`.
- A provided function `loss_fn(params)` computing the mean-squared-error loss of the network `yhat = tanh(X @ W1 + b1) @ w2 + b2`.

## Your task

Derive the analytic gradients of the loss with respect to **all four parameters** and compute them with NumPy (no autograd, no PyTorch):

```python
grads = {"W1": ..., "b1": ..., "w2": ..., "b2": ...}  # same shapes as params
```

Useful facts: `d/dz tanh(z) = 1 - tanh(z)^2`, and for `L = mean((yhat - y)^2)` the upstream gradient is `dL/dyhat = 2 (yhat - y) / n`.

## Evaluation

The evaluation cell runs a central-difference gradient check on every parameter. Maximum relative error must be below 1e-4.


In [ ]:
import numpy as np

rng = np.random.default_rng(8)
n, d, h = 200, 10, 16

X = rng.normal(size=(n, d))
y = rng.normal(size=n)
params = {
    "W1": rng.normal(0, 0.5, (d, h)),
    "b1": rng.normal(0, 0.1, h),
    "w2": rng.normal(0, 0.5, h),
    "b2": np.float64(0.3),
}

def loss_fn(p):
    hidden = np.tanh(X @ p["W1"] + p["b1"])
    yhat = hidden @ p["w2"] + p["b2"]
    return float(np.mean((yhat - y) ** 2))

print(f"loss at the given params: {loss_fn(params):.6f}")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: central-difference gradient check on every parameter
# =====================================================================
import numpy as np

if "grads" not in dir():
    raise NameError("Define the `grads` dict in the solution cell before running this cell.")

_eps = 1e-6
_worst = 0.0
print("=" * 52)
for _name in ["W1", "b1", "w2", "b2"]:
    _p = np.atleast_1d(np.asarray(params[_name], dtype=float)).copy()
    _g_user = np.atleast_1d(np.asarray(grads[_name], dtype=float))
    if _g_user.shape != _p.shape:
        raise ValueError(f"grads[{_name!r}] has shape {_g_user.shape}, expected {_p.shape}")
    _g_num = np.zeros_like(_p)
    _flat_p, _flat_g = _p.ravel(), _g_num.ravel()
    for _i in range(_flat_p.size):
        _orig = _flat_p[_i]
        _q = dict(params); _q[_name] = _p.reshape(np.shape(params[_name]))
        _flat_p[_i] = _orig + _eps; _lp = loss_fn(_q)
        _flat_p[_i] = _orig - _eps; _lm = loss_fn(_q)
        _flat_p[_i] = _orig
        _flat_g[_i] = (_lp - _lm) / (2 * _eps)
    _rel = np.abs(_g_user - _g_num) / np.maximum(1e-8, np.abs(_g_user) + np.abs(_g_num))
    _worst = max(_worst, float(_rel.max()))
    print(f"  {_name:<3} max relative error: {float(_rel.max()):.3e}")
print("=" * 52)
print(f"Worst relative error: {_worst:.3e}   (need < 1e-4)")
print("PASSED. Your derivation is correct!" if _worst < 1e-4 else "NOT PASSED yet. Recheck the chain rule shapes.")


## Solution

Forward pass: `Z = X @ W1 + b1`, `H = tanh(Z)`, `yhat = H @ w2 + b2`. Backward pass applies the chain rule layer by layer: the upstream gradient `2(yhat - y)/n` flows into `w2`/`b2` directly, then through `w2` and the `1 - tanh^2` derivative into `W1`/`b1`.

```python
import numpy as np

W1, b1, w2, b2 = params["W1"], params["b1"], params["w2"], params["b2"]
n = X.shape[0]

# Forward pass (keep intermediates for the backward pass)
Z = X @ W1 + b1        # (n, h)
H = np.tanh(Z)         # (n, h)
yhat = H @ w2 + b2     # (n,)

# Backward pass
dyhat = 2.0 * (yhat - y) / n       # dL/dyhat, (n,)
dH = np.outer(dyhat, w2)           # (n, h)
dZ = dH * (1.0 - H ** 2)           # tanh'(z) = 1 - tanh(z)^2

grads = {
    "W1": X.T @ dZ,                # (d, h)
    "b1": dZ.sum(axis=0),          # (h,)
    "w2": H.T @ dyhat,             # (h,)
    "b2": np.float64(dyhat.sum()), # scalar
}
```
